# Import danych do warstwy RAW

Pobieramy oryginalne pliki i zapisujemy je razem z metadanymi pobrania (URL, data, rozmiar, SHA-256). Źródła: GIOŚ (pomiary i metadane stacji) oraz Open-Meteo (pogoda).

Kod pobierania jest w `scripts/download_raw.py` i `scripts/download_openmeteo.py`.

## 1. Instalacja zależności

Komórka instaluje zależności w środowisku aktualnie wybranego kernela.

In [1]:
%pip install requests duckdb polars pyarrow scikit-learn

Note: you may need to restart the kernel to use updated packages.


## 2. Lokalizacja projektu

Notebook można uruchomić z katalogu głównego projektu albo z katalogu `notebooks`.

In [2]:
from pathlib import Path
import json
import runpy
import sys
import platform
from importlib.metadata import version

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
CONFIG_PATH = ROOT / 'config' / 'sources.json'
RAW_DIR = ROOT / 'data' / 'raw'
SCRIPT_PATH = ROOT / 'scripts' / 'download_raw.py'

if not CONFIG_PATH.is_file() or not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        'Uruchom notebook z katalogu projektu lub notebooks.'
    )

RAW_DIR.mkdir(parents=True, exist_ok=True)

print('Projekt:', ROOT)
print('Interpreter:', sys.executable)
print('Python:', platform.python_version())

for package in [
    'requests', 'duckdb', 'polars', 'pyarrow', 'scikit-learn'
]:
    print(f'{package}: {version(package)}')

Projekt: D:\project VS code\big_data\gios-bigdata
Interpreter: d:\project VS code\big_data\gios-bigdata\.venv\Scripts\python.exe
Python: 3.11.9
requests: 2.34.2
duckdb: 1.5.6
polars: 2.0.0
pyarrow: 26.0.0
scikit-learn: 1.9.1


## 3. Konfiguracja źródeł

Plik `config/sources.json` zawiera adresy plików GIOŚ. Wpisy z polem `year` trafiają do `data/raw/<rok>/`, a wpisy z polem `dataset` do `data/raw/<dataset>/`.

Źródło danych: GIOŚ - EKOINFONET (https://powietrze.gios.gov.pl/pjp/archives).

In [3]:
config = json.loads(
    CONFIG_PATH.read_text(encoding='utf-8-sig')
)

print(json.dumps(config, ensure_ascii=False, indent=2))

{
  "sources": [
    {
      "year": 2024,
      "url": "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582",
      "filename": "pomiary_2024.zip"
    }
  ]
}


## 4. Import danych GIOŚ do RAW

Pobieramy pomiary 2024 i metadane stacji. Poprawne, wcześniej pobrane pliki są pomijane.

In [4]:
import_module = runpy.run_path(str(SCRIPT_PATH))
import_module['main']()

2026-10-21 00:00:26,605 | INFO | Pobieranie: https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582
2026-10-21 00:00:34,264 | INFO | Zapisano: D:\project VS code\big_data\gios-bigdata\data\raw\2024\pomiary_2024.zip | 49.48 MB
2026-10-21 00:00:34,266 | INFO | Import do RAW zakończony.


[{'provider': 'GIOŚ',
  'layer': 'raw',
  'year': 2024,
  'source_url': 'https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582',
  'resolved_url': 'https://powietrze.gios.gov.pl/pjp/archives/downloadFile/582',
  'filename': 'pomiary_2024.zip',
  'downloaded_at_utc': '2026-10-20T22:00:34.220547+00:00',
  'size_bytes': 49482718,
  'sha256': '571dfa56866388c2904284ca6029bbf6016af3905b95bcacc5b3b6f6fa2d00e1',
  'content_type': 'application/zip'}]

## 5. Pogoda - Open-Meteo

GIOŚ nie udostępnia danych pogodowych, więc pobieramy je z Open-Meteo (licencja CC BY 4.0) dla każdej stacji mierzącej PM2.5 w 2024 r. Współrzędne bierzemy z metadanych stacji, dlatego ta sekcja jest po sekcji 4. Czas jest w UTC.

Między zapytaniami jest 2 s przerwy, więc pierwsze pobranie trwa kilka minut.

In [1]:
OPENMETEO_SCRIPT = ROOT / 'scripts' / 'download_openmeteo.py'

openmeteo_module = runpy.run_path(str(OPENMETEO_SCRIPT))
openmeteo_results = openmeteo_module['main']()

NameError: name 'ROOT' is not defined

## 6. Podsumowanie lokalnych danych RAW

Liczba i rozmiar pobranych plików w każdym folderze.

In [ ]:
records = []

for metadata_path in sorted(
    RAW_DIR.rglob('*.metadata.json')
):
    metadata = json.loads(
        metadata_path.read_text(encoding='utf-8')
    )
    data_path = metadata_path.parent / metadata['filename']
    if data_path.is_file():
        metadata['folder'] = metadata_path.parent.relative_to(RAW_DIR).as_posix()
        records.append(metadata)

total_bytes = sum(item['size_bytes'] for item in records)

print('Liczba plików:', len(records))
print(f'Łączny rozmiar: {total_bytes / 1_000_000:.2f} MB')
print()

summary = {}

for item in records:
    folder = summary.setdefault(
        item['folder'],
        {'zrodlo': item['provider'], 'pliki': 0, 'bajty': 0},
    )
    folder['pliki'] += 1
    folder['bajty'] += item['size_bytes']

for name, info in summary.items():
    print(
        f"{name:18} {info['zrodlo']:12} "
        f"plików: {info['pliki']:4}   {info['bajty'] / 1_000_000:7.2f} MB"
    )

## 7. Wnioski z etapu RAW

Jakie lata i pliki pobrano?

Rok 2024. Z GIOŚ archiwum pomiarów `pomiary_2024.zip` (29 plików XLSX) i metadane stacji `metadane_stacji.xlsx`. Z Open-Meteo 96 plików JSON z pogodą, po jednym dla każdej stacji mierzącej PM2.5.

Jaki jest rozmiar danych?

98 plików, razem 89,3 MB: pomiary GIOŚ 49,5 MB, metadane stacji 0,6 MB, pogoda 39,2 MB.

Czy wystąpiły błędy pobierania?

Nie. Wszystkie pliki pobrały się poprawnie, a przy ponownym uruchomieniu są sprawdzane sumą SHA-256 i pomijane.

Jaki zakres danych wybrano do dalszej analizy?

Cały rok 2024: wszystkie wskaźniki zanieczyszczeń poza depozycją (8 952 726 pomiarów). Celem modelu jest godzinowe PM2.5 z 96 stacji, a dla tych stacji mamy godzinową pogodę (temperatura, wilgotność, opad, ciśnienie, wiatr).

Następny etap: notebooki `02_bronze_gios.ipynb` i `02_bronze_openmeteo.ipynb`.